# Electron Reconstruction Efficiency — Tag-and-Probe Analysis

**Based on:**
- S. Younas, *Rom. J. Phys.* **68**, 403 (2023) — methodology and uncertainty study  
  [Full text PDF](https://www.nipne.ro/rjp/2023_68_7-8/RomJPhys.68.403.pdf)
- ATLAS Collaboration, *Eur. Phys. J. C* **79**, 639 (2019) — efficiency measurement style  
  [DOI](https://doi.org/10.1140/epjc/s10052-019-7140-6) | [arXiv:1902.04655](https://arxiv.org/abs/1902.04655)

---

## What Is Tag-and-Probe?

The **Tag-and-Probe (T&P)** method measures electron reconstruction efficiency
directly from data using Z → ee decays without relying on simulation.

- **Tag**: one electron passing **tight** selection (confirms a genuine Z event)
- **Probe**: the other electron — efficiency is measured on these
- **Efficiency**: fraction of probes passing the reconstruction criterion
- **Scale Factor**: ratio of data efficiency to MC efficiency

### ATLAS Detector Regions (EPJC 2019, Section 2)

| Region | \|η\| range | Notes |
|--------|------------|-------|
| **Barrel** | < 1.37 | High efficiency ~96–97% |
| **Transition** | 1.37 – 1.52 | Excluded — large inactive material |
| **Endcap** | 1.52 – 2.47 | Slightly lower efficiency ~90–91% |

> **Note:** ATLAS transition region is 1.37 < |η| < 1.52. This differs from CMS (1.4442–1.566).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

# ATLAS definitions (EPJC 2019, Section 2)
BARREL_MAX = 1.37   # barrel / transition boundary
TRANS_MAX  = 1.52   # transition / endcap boundary
ACCEPTANCE = 2.47   # outer precision measurement limit

print('ATLAS EM calorimeter regions:')
print(f'  Barrel:     |eta| < {BARREL_MAX}  (~96-97% efficiency)')
print(f'  Transition: {BARREL_MAX} < |eta| < {TRANS_MAX}  (excluded)')
print(f'  Endcap:     {TRANS_MAX} < |eta| < {ACCEPTANCE}  (~90-91% efficiency)')

## Step 1: Generate Synthetic Z→ee Dataset

We generate synthetic probe electrons matching the statistical properties
of real ATLAS Z→ee data. No proprietary CERN data is used.

**True efficiency model:**
- ET turn-on: logistic function rising from ~93% at 15 GeV to plateau ~97% at 30+ GeV
- Eta dependence: barrel ~96.8%, transition ~65%, endcap ~90.6%
- Based on published ATLAS Run 2 values (EPJC 2019)

In [ ]:
def true_efficiency_data(et, eta):
    """Data efficiency model based on ATLAS EPJC 2019 published values."""
    et_factor = 1.0 / (1.0 + np.exp(-(et - 25.0) / 4.0))
    ae = np.abs(eta)
    eta_eff = np.where(
        ae < BARREL_MAX,  0.968,
        np.where(ae < TRANS_MAX,  0.55 + 0.12 * et_factor,
        np.where(ae < ACCEPTANCE, 0.906, 0.0)))
    return et_factor * eta_eff

def true_efficiency_mc(et, eta):
    """MC efficiency: slightly overestimates data (drives scale factors < 1)."""
    ae = np.abs(eta)
    corr = np.where(ae < BARREL_MAX, 1.009,
           np.where(ae < TRANS_MAX,  0.940, 1.004))
    return np.clip(true_efficiency_data(et, eta) * corr, 0, 1)

# Generate datasets
N = 50000
rng = np.random.default_rng(42)

et  = np.clip(np.abs(rng.normal(45.0, 15.0, N)) + 5.0, 15.0, 150.0)
eta = rng.uniform(-ACCEPTANCE, ACCEPTANCE, N)

# Tag selection: ET > 27 GeV, outside transition region
tag_et   = rng.exponential(35.0, N) + 27.0
tag_eta  = rng.uniform(-ACCEPTANCE, ACCEPTANCE, N)
in_trans = (np.abs(tag_eta) > BARREL_MAX) & (np.abs(tag_eta) < TRANS_MAX)
tag_pass = (tag_et > 27.0) & (~in_trans)

# Probe pass based on true efficiency
eff      = true_efficiency_data(et, eta)
probe_d  = rng.random(N) < eff

# MC dataset
rng2 = np.random.default_rng(99)
et_m  = np.clip(np.abs(rng2.normal(45.0, 15.0, N)) + 5.0, 15.0, 150.0)
eta_m = rng2.uniform(-ACCEPTANCE, ACCEPTANCE, N)
tag_m = (rng2.exponential(35.0, N) + 27.0 > 27.0) & \
        (~((np.abs(rng2.uniform(-ACCEPTANCE, ACCEPTANCE, N)) > BARREL_MAX) &
           (np.abs(rng2.uniform(-ACCEPTANCE, ACCEPTANCE, N)) < TRANS_MAX)))
probe_m = rng2.random(N) < true_efficiency_mc(et_m, eta_m)

df_d = pd.DataFrame({'et': et,   'eta': eta,   'tag': tag_pass.astype(int), 'probe': probe_d.astype(int)})
df_m = pd.DataFrame({'et': et_m, 'eta': eta_m, 'tag': tag_m.astype(int),   'probe': probe_m.astype(int)})

print(f'Data: {N:,} events | tagged: {tag_pass.sum():,} | probe pass: {probe_d.sum():,}')
print(f'MC:   {N:,} events | tagged: {tag_m.sum():,}   | probe pass: {probe_m.sum():,}')

## Step 2: Tag-and-Probe Selection

Apply ATLAS selection criteria (Section 4.1, EPJC 2019):

| Criterion | Tag | Probe |
|-----------|-----|-------|
| ET | > 27 GeV | > 15 GeV |
| |η| | < 2.47, **outside transition** | < 2.47 |
| Selection | Tight ID + isolation + trigger | EM-cluster only |
| Z mass | 75 < m_ee < 105 GeV | — |

In [ ]:
# Apply tag requirement
tagged_d = df_d[df_d['tag'] == 1]
tagged_m = df_m[df_m['tag'] == 1]

# ET and eta bins
ET_BINS  = [15, 20, 25, 30, 40, 60, 80, 150]
ETA_BINS = [-2.47,-2.01,-1.52, -1.37,-1.01,-0.60,-0.20,
              0.00,  0.20,  0.60,  1.01,  1.37, 1.52, 2.01, 2.47]
ET_CENTRES  = [(ET_BINS[i]+ET_BINS[i+1])/2  for i in range(len(ET_BINS)-1)]
ETA_CENTRES = [(ETA_BINS[j]+ETA_BINS[j+1])/2 for j in range(len(ETA_BINS)-1)]

print(f'Tagged data probes: {len(tagged_d):,}')
print(f'Tagged MC probes:   {len(tagged_m):,}')
print(f'ET bins:  {ET_BINS}')
print(f'Eta bins: {len(ETA_BINS)-1} bins from {ETA_BINS[0]} to {ETA_BINS[-1]}')

## Step 3: Compute Efficiency with Clopper-Pearson Uncertainty

Statistical uncertainty uses the **exact binomial (Clopper-Pearson) interval**
at 68.3% confidence level, as recommended in EPJC 2019 Section 4:

$$\varepsilon = \frac{N_{\rm pass}}{N_{\rm probe}}, \quad
\sigma_{\rm stat} = \frac{\Delta_{\rm hi} - \Delta_{\rm lo}}{2}$$

This correctly handles low-statistics bins (transition region, high-ET bins)
unlike the naive approximation $\sqrt{\varepsilon(1-\varepsilon)/n}$.

In [ ]:
def clopper_pearson(k, n, cl=0.683):
    """Exact binomial interval at 68.3% CL (1-sigma)."""
    if n == 0: return np.nan, np.nan, np.nan
    alpha = 1 - cl
    lo = float(beta_dist.ppf(alpha/2,   max(k,   1e-9), max(n-k+1, 1e-9)))
    hi = float(beta_dist.ppf(1-alpha/2, max(k+1, 1e-9), max(n-k,   1e-9)))
    lo, hi = np.clip(lo, 0, 1), np.clip(hi, 0, 1)
    return k/n, k/n-lo, hi-k/n

def compute_eff_1d(df, bins, var):
    """Efficiency in 1D bins of var ('et' or 'eta')."""
    out = []
    for i in range(len(bins)-1):
        m   = (df[var] >= bins[i]) & (df[var] < bins[i+1])
        k,n = df['probe'][m].sum(), m.sum()
        e,el,eh = clopper_pearson(k, n)
        out.append({'bin_lo': bins[i], 'bin_hi': bins[i+1],
                    'centre': (bins[i]+bins[i+1])/2,
                    'eff': e, 'err_lo': el, 'err_hi': eh, 'n': n})
    return pd.DataFrame(out)

df_eff_et_d  = compute_eff_1d(tagged_d, ET_BINS,  'et')
df_eff_et_m  = compute_eff_1d(tagged_m, ET_BINS,  'et')
df_eff_eta_d = compute_eff_1d(tagged_d, ETA_BINS, 'eta')
df_eff_eta_m = compute_eff_1d(tagged_m, ETA_BINS, 'eta')

print('Efficiency vs ET (data):')
print(df_eff_et_d[['centre','eff','err_lo','err_hi','n']].to_string(
      index=False, float_format='{:.4f}'.format))

## Step 4: Systematic Uncertainties

Three systematic variations from the Romanian Journal paper (Section 2.4):

1. **Tag tightening**: require tag ET > 30 GeV (tighter than nominal 27 GeV)
2. **Background subtraction**: remove 5% of probes (simulates background variation)
3. **ET reweighting**: weight probes by 1/ET (tests fit window dependence)

Total: $\sigma_{\rm syst} = \sqrt{\sum_i (\Delta\varepsilon_i)^2}$

In [ ]:
rng3 = np.random.default_rng(77)
nom  = df_eff_et_d['eff'].values

# Tight tag
df_tight = df_d[(df_d['tag']==1) & (df_d['et']>30)]
v1 = compute_eff_1d(df_tight, ET_BINS, 'et')['eff'].values

# Background -5%
df_bg = tagged_d[rng3.random(len(tagged_d)) > 0.05]
v2 = compute_eff_1d(df_bg, ET_BINS, 'et')['eff'].values

# ET reweight
df_rw = tagged_d.copy()
df_rw['_w'] = 1.0 / (df_rw['et'] + 1.0)
v3 = compute_eff_1d(df_rw, ET_BINS, 'et')['eff'].values

syst = np.sqrt((v1-nom)**2 + (v2-nom)**2 + (v3-nom)**2)
stat = df_eff_et_d['err_lo'].values
total = np.sqrt(stat**2 + syst**2)

df_unc = pd.DataFrame({'ET_bin': [f'{ET_BINS[i]}-{ET_BINS[i+1]}' for i in range(len(ET_BINS)-1)],
                        'eff': nom, 'stat': stat, 'syst': syst, 'total': total})
print('Uncertainty breakdown by ET bin:')
print(df_unc.to_string(index=False, float_format='{:.5f}'.format))

## Step 5: 2D Efficiency Map (ET × η)

In [ ]:
n_et  = len(ET_BINS)  - 1
n_eta = len(ETA_BINS) - 1
eff_map = np.full((n_et, n_eta), np.nan)

for i in range(n_et):
    for j in range(n_eta):
        m = ((tagged_d['et']  >= ET_BINS[i])  & (tagged_d['et']  < ET_BINS[i+1]) &
             (tagged_d['eta'] >= ETA_BINS[j]) & (tagged_d['eta'] < ETA_BINS[j+1]))
        k, n = tagged_d['probe'][m].sum(), m.sum()
        if n > 0: eff_map[i, j], *_ = clopper_pearson(k, n)

fig, ax = plt.subplots(figsize=(13, 5))
masked = np.ma.masked_invalid(eff_map)
im = ax.imshow(masked, origin='lower', aspect='auto',
               vmin=0.60, vmax=1.0, cmap='RdYlGn',
               extent=[ETA_BINS[0], ETA_BINS[-1], 0, n_et])
plt.colorbar(im, ax=ax, label='Efficiency', shrink=0.85)

# Cell annotations
for i in range(n_et):
    for j in range(n_eta):
        if not np.isnan(eff_map[i, j]):
            xpos = ETA_CENTRES[j]
            ypos = i + 0.5
            color = 'black' if eff_map[i,j] > 0.82 else 'white'
            ax.text(xpos, ypos, f'{eff_map[i,j]:.2f}',
                    ha='center', va='center', fontsize=5.5, color=color)

for tl, th in [(-TRANS_MAX,-BARREL_MAX),(BARREL_MAX,TRANS_MAX)]:
    ax.axvspan(tl, th, alpha=0.25, color='gray')

et_labels = [f'{ET_BINS[i]}-{ET_BINS[i+1]}' for i in range(n_et)]
ax.set_yticks(np.arange(n_et)+0.5)
ax.set_yticklabels(et_labels, fontsize=8)
ax.set_xlabel(r'$\eta$', fontsize=13)
ax.set_ylabel(r'$E_T$ [GeV]', fontsize=12)
ax.set_title(r'2D Electron Reconstruction Efficiency $\varepsilon(E_T, \eta)$\n'
             r'Grey shading = ATLAS transition region (1.37 < |$\eta$| < 1.52, excluded)', fontsize=11)
plt.tight_layout()
plt.savefig('../results/efficiency_2d_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/efficiency_2d_map.png')

## Step 6: Scale Factors

Scale factors correct simulation to match data:
$$SF(E_T, \eta) = \frac{\varepsilon_{\rm data}(E_T, \eta)}{\varepsilon_{\rm MC}(E_T, \eta)}$$

In [ ]:
sf_et  = df_eff_et_d['eff'].values  / df_eff_et_m['eff'].values
sf_eta = df_eff_eta_d['eff'].values / df_eff_eta_m['eff'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, centres, sf, bins, xlabel, xlog in [
    (axes[0], ET_CENTRES,  sf_et,  ET_BINS,  r'$E_T$ [GeV]', True),
    (axes[1], ETA_CENTRES, sf_eta, ETA_BINS, r'$\eta$',       False),
]:
    ax.plot(centres, sf, 'o', color='black', ms=6, mfc='white', mew=1.5)
    ax.axhline(y=1.0, color='red', lw=1.2)
    ax.fill_between([min(bins), max(bins)], [0.97,0.97], [1.03,1.03], alpha=0.12, color='green')
    ax.set_ylim(0.96, 1.04)
    ax.set_ylabel('Scale Factor  (Data/MC)', fontsize=11)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.grid(True, alpha=0.2)
    if xlog: ax.set_xscale('log')
    if not xlog:
        for tl, th in [(-TRANS_MAX,-BARREL_MAX),(BARREL_MAX,TRANS_MAX)]:
            ax.axvspan(tl, th, alpha=0.18, color='orange')

axes[0].set_title('Scale Factors vs ET')
axes[1].set_title('Scale Factors vs η')
plt.suptitle(r'Data/MC Efficiency Scale Factors — $Z\rightarrow ee$', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../results/scale_factors_1d.png', dpi=150, bbox_inches='tight')
plt.show()
print('Scale factors vs ET:', np.round(sf_et, 4))

## Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Romanian Journal paper — methodology reference | ✅ |
| 2 | Synthetic Z→ee dataset (50,000 probe electrons) | ✅ |
| 3 | Tag-and-Probe selection (ATLAS criteria) | ✅ |
| 4 | Systematic uncertainties (3 variations) | ✅ |
| 5 | 2D efficiency map ε(ET, η) | ✅ |
| 6 | Data/MC scale factors | ✅ |

**Key results:**
- Barrel efficiency: ~96.8% (Data), ~97.8% (MC), **SF ≈ 0.990**
- Endcap efficiency: ~90.6% (Data), ~91.0% (MC), **SF ≈ 0.996**
- Transition region: excluded from measurement (inactive material)

**References:**
1. ATLAS EPJC 79, 639 (2019) — [DOI](https://doi.org/10.1140/epjc/s10052-019-7140-6) | [arXiv:1902.04655](https://arxiv.org/abs/1902.04655)
2. S. Younas, Rom. J. Phys. 68, 403 (2023) — [Full text](https://www.nipne.ro/rjp/2023_68_7-8/RomJPhys.68.403.pdf)